# Onset-aligned time courses of the first three PCs
Project neural activity around movement onset onto the **saved tracking PCA axes** and average scores across bouts. Red = tracking; black = playback. Columns show PC1-PC3; rows compare raw and baseline-subtracted neural activity. Shading is SEM across complete bouts, not neurons or independent animals.

Each row uses its own saved tracking PCA model: tracking and playback share identical loadings and centering within that row. PC identities/signs can differ between the raw and corrected fits, so the rows are not the same coordinate system. These are **PC score time courses**, not single-neuron firing-rate PSTHs. Do not average PC1 across sessions, whose loadings and signs are unrelated.

Baseline subtraction means subtracting each bout/neuron's raw mean rate in **[-1, -0.7) s**, before smoothing and projection. The PCA training mean is then subtracted as required by the saved transform. A corrected PC trace therefore need not have zero baseline: no additional PC-level re-zeroing is applied.

Saved PCAs were fitted on five 100 ms windows over [-0.5, 0) s. This notebook keeps those axes fixed and projects finer time bins before and after onset; it does not fit a new PCA on the plotted interval. Post-onset activity is outside the PCA fitting interval and can include movement and sensory responses.


A small bar plot under each time course shows one signed PCA loading per unit: **x = units, y = weights**. Units remain in the original recording/PCA input order across all six panels; tick labels use the neuron ID suffix. Common symmetric weight limits allow comparison of positive and negative loadings. No PCA or PSTH recomputation is needed to change this display.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import reload
import hashlib
import json
import pickle
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
cwd = Path.cwd().resolve()
repo_dir = next((candidate for parent in (cwd, *cwd.parents) for candidate in (parent, parent / "etheremin_project")
                 if (candidate / "movement_bouts" / "pc_psth.py").is_file()), None)
if repo_dir is None:
    raise FileNotFoundError("Run from the workspace or inside the repository.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
import pc_psth
reload(pc_psth)
from pc_psth import project_pc_psths, plot_pc_psths
DATA_DIR = repo_dir.parent / "data"
PREMOTOR_DIR = DATA_DIR / "derived/premotor_activity"


## Settings
Use the same native 5 ms bins and 10 ms Gaussian smoothing as the single-neuron PSTH notebook. Smoothing is symmetric and can spread changes across onset; set sigma to zero to disable it. The wider extraction includes baseline and smoothing margins; only [-0.5, +0.5) s is shown initially. Change settings and rerun below them.


In [ ]:
SESSION_IDX = 0
DIRECTION = None  # None=all; -1=leftward; +1=rightward.
WINDOW_S = (-1.2, 1.0)
DISPLAY_WINDOW_S = (-0.5, 0.5)
BIN_WIDTH_S = 0.005
SMOOTHING_SIGMA_S = 0.01
N_COMPONENTS = 3
PCA_PATHS = {
    "raw": PREMOTOR_DIR / "runs/analysis_-0.5_0_5bins_raw/pca/pca_5bins.pkl",
    "baseline_subtracted": PREMOTOR_DIR / "runs/analysis_-0.5_0_5bins_baseline_subtracted_-1_-0.7/pca/pca_5bins.pkl",
}

CACHE_SESSION_INDICES = None  # None=all eligible sessions; or e.g. [0, 1, 2].


## Verify saved fits and load source metadata
Read original recordings to include the post-onset interval, which is not in the premotor extraction. Only one animal/headstage collection is cached at a time. Check exact recorded neuron IDs/order against each tracking model and preserve original movement-bout IDs. Conditions are assigned at onset; transition-crossing windows are not separately filtered. Bounds failures are excluded and counted identically in both rows.


In [ ]:
bout_dir = DATA_DIR / "derived/movement_bouts"
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
pca_bundles = {mode: pd.read_pickle(path) for mode, path in PCA_PATHS.items()}
pca_hashes = {mode: hashlib.sha256(path.read_bytes()).hexdigest() for mode, path in PCA_PATHS.items()}
for mode, bundle in pca_bundles.items():
    settings = bundle["settings"]
    if settings["activity_mode"] != mode or settings.get("standardized", False) or settings.get("whiten", False):
        raise ValueError("Use unstandardized, unwhitened saved PCAs in the specified activity mode.")
    for name, expected in settings["extraction_parameters"]["source_sha256"].items():
        if hashlib.sha256((bout_dir / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"Source {name} differs from saved PCA; regenerate aligned PCA fits first.")
    for field in ("DT", "N_BINS", "analysis_window_s"):
        if settings[field] != pca_bundles["raw"]["settings"][field]:
            raise ValueError(f"Raw/corrected saved PCA preprocessing differs in {field}.")
DT = pca_bundles["raw"]["settings"]["DT"]
BASELINE_WINDOW_S = tuple(pca_bundles["baseline_subtracted"]["settings"]["baseline_window_s"])
for bundle in pca_bundles.values():
    expected_baseline_end = pca_bundles["baseline_subtracted"]["settings"]["analysis_window_s"][0]
    if BASELINE_WINDOW_S[1] > expected_baseline_end:
        raise ValueError("Baseline must end before the PCA analysis interval starts.")
eligible_indices = [idx for idx in session_info.index
                    if all(idx in b["sessions"] and 0 in b["sessions"][idx]
                           and b["sessions"][idx][0]["model"].n_components_ >= N_COMPONENTS
                           for b in pca_bundles.values())]
if SESSION_IDX not in eligible_indices:
    raise ValueError("Selected session does not have both saved tracking PCAs with enough components.")
recording_cache = {}

def load_recording(idx):
    info = session_info.loc[idx]
    source = info["feature_file"].replace("_feature_ss", "_data_ss")
    if recording_cache.get("source") != source:
        recording_cache.clear()
        print(f"Loading {source}")
        with (DATA_DIR / source).open("rb") as stream:
            recordings = pickle.load(stream)
        ids_file = info["feature_file"].replace("_feature_ss", "_neuron_idx_ss")
        with (DATA_DIR / ids_file).open("rb") as stream:
            ids = pickle.load(stream)
        recording_cache.update(source=source, recordings=recordings, ids=ids)
    session = int(info["session"])
    neural, ids = recording_cache["recordings"][session], recording_cache["ids"][session]
    if neural.shape != (len(ids), int(info["n_bins"])):
        raise ValueError("Original recording shape differs from saved session metadata.")
    return neural, ids


def compute_session_pcs(idx, direction=None):
    if direction not in (None, -1, 1):
        raise ValueError("Choose all, leftward or rightward movements.")
    neural, ids = load_recording(idx)
    bouts = movement_bouts.loc[movement_bouts["idx"].eq(idx) & movement_bouts["condition"].isin([0, 1])].sort_values("onset_bin")
    if direction is not None:
        bouts = bouts.loc[bouts["initial_direction"].eq(direction)]
    mode_fits = {mode: b["sessions"][idx][0]["model"] for mode, b in pca_bundles.items()}
    result = project_pc_psths(neural, ids, bouts, mode_fits, dt=DT,
        window_s=WINDOW_S, bin_width_s=BIN_WIDTH_S, baseline_window_s=BASELINE_WINDOW_S,
        smoothing_sigma_s=SMOOTHING_SIGMA_S, n_components=N_COMPONENTS)
    result["settings"].update(idx=int(idx), direction=direction, pca_paths={k: str(v) for k, v in PCA_PATHS.items()},
                              pca_sha256=pca_hashes, display_window_s=list(DISPLAY_WINDOW_S),
                              pca_fit_settings={mode: b["settings"] for mode, b in pca_bundles.items()})
    result["session_info"] = session_info.loc[[idx]].copy()
    return result


def draw_session_pcs(result):
    idx = result["settings"]["idx"]
    info = session_info.loc[idx]
    title = f"idx={idx} | {info.animal} hs{info.headstage} | session {info.session} | direction={result['settings']['direction']}"
    fig, axes = plot_pc_psths(result, display_window_s=DISPLAY_WINDOW_S, title=title)
    print(f"{result['n_complete_bouts']} complete bouts; {result['n_excluded_bouts']} excluded at recording bounds.")
    return fig, axes
print(f"{len(eligible_indices)} sessions have both saved tracking PCAs. Baseline: {BASELINE_WINDOW_S} s.")


## Precompute once, then browse from memory
Load each original animal/headstage collection once, project each session once for both preprocessing modes, then summarize **all/left/right** bouts. Retain only means, SEMs, bout counts/IDs and PC weights/models. Discard per-bout score arrays and release the original recordings afterward.

The first pass takes longer. Rerunning this cell reuses completed sessions when the settings match; changed projection settings invalidate the cache. No result files or PNGs are written. Run this cell before the static figure and slider.


In [ ]:
from tqdm.auto import tqdm
reload(pc_psth)
from pc_psth import project_pc_psths, summarize_pc_psths, plot_pc_psths


def current_pc_cache_settings():
    return {"dt": DT, "window_s": list(WINDOW_S), "bin_width_s": BIN_WIDTH_S,
            "baseline_window_s": list(BASELINE_WINDOW_S), "smoothing_sigma_s": SMOOTHING_SIGMA_S,
            "n_components": N_COMPONENTS, "pca_sha256": dict(pca_hashes),
            "pca_paths": {mode: str(path) for mode, path in PCA_PATHS.items()},
            "session_indices": list(cached_indices)}

cached_indices = list(eligible_indices if CACHE_SESSION_INDICES is None else CACHE_SESSION_INDICES)
if not cached_indices or len(set(cached_indices)) != len(cached_indices) or not set(cached_indices).issubset(eligible_indices):
    raise ValueError("Choose nonempty, unique sessions with both tracking PCA bases.")
if SESSION_IDX not in cached_indices:
    raise ValueError("Include SESSION_IDX in the precomputed session selection.")
cache_settings = current_pc_cache_settings()
if "pc_psth_cache" not in globals() or pc_psth_cache["settings"] != cache_settings:
    pc_psth_cache = {"settings": cache_settings, "sessions": {}}
ordered_indices = [idx for _, rows in session_info.loc[cached_indices].groupby("feature_file", sort=False)
                   for idx in rows.index]
for idx in tqdm(ordered_indices, desc="Precomputing PC PSTHs"):
    if idx in pc_psth_cache["sessions"]:
        continue
    full_result = compute_session_pcs(idx, direction=None)
    pc_psth_cache["sessions"][idx] = summarize_pc_psths(full_result)
    del full_result
recording_cache.clear()  # Release the last large source collection after precomputation.


def get_cached_pc_psth(idx, direction=None):
    if current_pc_cache_settings() != pc_psth_cache["settings"]:
        raise ValueError("PSTH settings changed; rerun the precompute cell before browsing.")
    if direction not in (None, -1, 1) or idx not in pc_psth_cache["sessions"]:
        raise ValueError("Choose a cached session and all/left/right direction.")
    return pc_psth_cache["sessions"][idx][direction]

summary_rows = []
summary_bytes = 0
for idx, directions in pc_psth_cache["sessions"].items():
    for direction, result in directions.items():
        summary_rows.append({"idx": idx, "direction": direction if direction is not None else "all",
                             "n_complete_bouts": result["n_complete_bouts"], "n_excluded_bouts": result["n_excluded_bouts"]})
        for values in result["modes"].values():
            for condition in values["conditions"].values():
                summary_bytes += condition["mean"].nbytes + condition["sem"].nbytes
cache_summary = pd.DataFrame(summary_rows).set_index(["idx", "direction"])
print(f"Cached {len(pc_psth_cache['sessions'])} sessions and all/left/right views; mean/SEM arrays: {summary_bytes/2**20:.1f} MiB.")
print("Source recordings released. Slider/direction callbacks now only redraw cached summaries.")
display(cache_summary.head(9))


## First session: PC1-PC3, raw and baseline-subtracted
This static figure remains visible without widget support. It uses only the precomputed mean/SEM and loading summaries below; no original recordings are loaded while plotting.


In [ ]:
reload(pc_psth)
from pc_psth import plot_pc_psths

pc_psth_result = get_cached_pc_psth(SESSION_IDX, DIRECTION)
fig_pc_psth, axes_pc_psth = draw_session_pcs(pc_psth_result)
display(fig_pc_psth)
plt.close(fig_pc_psth)


## Browse sessions and directions
The session slider redraws when released; direction changes redraw immediately. Both controls read the in-memory cache: no recording loads, projection, filtering of per-bout arrays or PCA fitting occur in their callbacks. Each session keeps its own fixed tracking bases. Recompute the cache after changing binning, baseline, smoothing or PCA models. Display limits can change without recomputing within the cached extraction interval. Restarting the kernel clears the cache; nothing is saved to disk.


In [ ]:
reload(pc_psth)
from pc_psth import plot_pc_psths

session_selector = widgets.SelectionSlider(
    options=[(str(idx), idx) for idx in cached_indices],
    value=SESSION_IDX, description="Session:", continuous_update=False,
    layout=widgets.Layout(width="650px"))
session_details = widgets.HTML()

direction_selector = widgets.Dropdown(options=[("All directions", None), ("Leftward", -1), ("Rightward", 1)],
                                     value=DIRECTION, description="Direction:")
plot_output = widgets.Output()
viewer_state = {"result": pc_psth_result}

def redraw_pc_psth(change=None):
    with plot_output:
        clear_output(wait=True)
        idx, direction = session_selector.value, direction_selector.value
        info = session_info.loc[idx]
        session_details.value = f"<b>idx={idx}</b> | {info.animal} | hs{info.headstage} | recording session {info.session}"
        viewer_state["result"] = get_cached_pc_psth(idx, direction)
        figure, _ = draw_session_pcs(viewer_state["result"])
        display(figure)
        plt.close(figure)

session_selector.observe(redraw_pc_psth, names="value")
direction_selector.observe(redraw_pc_psth, names="value")
display(widgets.VBox([session_selector, session_details, direction_selector, plot_output]))
redraw_pc_psth()
